In [ ]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[2, 4, 6, 11, 13, 15, 18, 20, 22, 24, 26, 29])

nb_name = "0C_PILOT_trajectory.ipynb"

_compare_version(nb_name)

# Trajectory analysis with PILOT
<hr style="border:2px solid black"> </hr>

## 1.  Description

**Welcome to our extented notebook for scRNA data for unsupervised trajectory analysis with PILOT:**

PILOT is a python library developed by Costalab to perform patient-level analysis by using optimal transport. The objective is to enable the direct comparison between multiple scales samples e.g. control vs. disease and the observation of gene expression at different timepoints.
    
    
**Reference**: Joodaki et al. (2024) Detection of PatIent-Level distances from single cell genomics and pathomics data with   
    Optimal Transport (PILOT), Molecular Systems Biology, Volume 20, [10.1038/s44320-023-00003-8](https://doi.org/10.1038/s44320-023-00003-8)

<div class="alert alert-block alert-danger">
    
**Requires a clustered or otherwise categorized anndata object. A clustering can be generated with a clustering notebook (e.g. <code>rna_analysis/notebooks/04_clustering.ipynb</code>).**
    
**Move this notebook into the notebook folder (e.g. <code>rna_analysis/notebooks/</code>) of the respective analysis before using it!**

</div>

-----------

## 2. Setup

### 2.1 Loading packages

In [ ]:
import scanpy as sc
import pandas as pd
import pilotpy as pl

import sctoolbox.utils as utils

from warnings import warn
from sctoolbox import settings

settings.settings_from_config("config.yaml", key="PILOT_trajectory")

-----------

## 3. Load anndata

### 3.1 Loading anndata

<div class="alert alert-block alert-danger">
    
**Pre-analyzed (clustered) data can be imported using the Assembly notebook, followed by, for example, this notebook to conduct analysis.**
    
</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
clustered_adata = "anndata_4.h5ad"

In [ ]:
adata = utils.adata.load_h5ad(clustered_adata)

with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata)
    display(adata.obs)
    display(adata.var)
    display(adata.obsm)

### 3.2 Prevent division-by-zero in fold change (FC) calculations using MAGIC 

The absence of mRNA molecules in PILOT can lead to incorrect downstream FC calculations on account of division-by-zero artifacts. To avoid this, the MAGIC (Markov Affinity-based Graph Imputation of Cells) algorithm is applied to denoise the cell count matrix and supplement missing transcripts through data diffusion for all genes across cells with very small positive values.

Reference: Van Dijk et al. Recovering gene interactions from single-cell data using data diffusion. Cell, 174(3):716–729.e27, jul 2018, [10.1016/j.cell.2018.05.061](https://doi.org/10.1016/j.cell.2018.05.061)

| **Parameter** | **Description** |  **Options** |
|:---|:---|:---|
| <code>**name_list**</code> | Which genes to denoise. | The default <code>all_genes</code> may require a large amount of memory if the input data is sparse. Pass a list of genes instead, if desired. Another possibility is <code>'pca_only'</code>,  which denoises the PCA scores of the PCA components instead of genes. |
| <code>**t**</code> | Number of diffusion steps (power of the diffusion operator). Higher values increase smoothing.| If <code>auto</code>, t is selected according to the [Procrustes disparity](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.procrustes.html) of the diffused data.<br>**Practical guidance for <code>t</code>:**<br> **<code>t</code> = 1–2**: Minimal smoothing, Preserves sharp cell-state differences, Good for subtle trajectories<br>**<code>t</code> = 3–4**: Standard choice, Balances noise reduction and structure<br>**<code>t</code> ≥ 5**: Strong smoothing, Can blur clusters and inflate correlations, Often problematic for downstream analyses|
| <code>**solver**</code> | Which solver to use. | <code>exact</code> uses the implementation described in van Dijk et al. [2018]. <code>approximate</code> uses a faster implementation that performs imputation in the PCA space and then projects back to the gene space. Note, the <code>approximate</code> solver may return negative values.|

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Which genes to denoise
name_list = 'all_genes'

# time-steps to use for diffusion
t = 2

# Which solver to use
solver = 'exact'

In [ ]:
sc.external.pp.magic(adata,
                     name_list=name_list,
                     t=t,
                     solver=solver)

-----------

## 4. General input

### 4.1 Input

<div class="alert alert-block alert-danger">

**The column for the <code>sample</code> parameter needs at least five different categories to calculate the diffusionmap.**

</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Provide the name of the variable in the obsm level of your Anndata that holds the dimension reduction (PCA representation).
emb_matrix = 'X_pca'

# Specify the name of the column in the obs level that corresponds to cell types or cell clusters.
clusters = 'celltypes'

# Indicate the column name in the obs level that contains information about samples or patients.
sample = 'Sample'

# Provide the column name in the obs level that represents the status or disease (e.g., “control” or “case”).
status = 'timepoint'

# Add the cell types/cell clusters to remove, which may be necessary e.g. in cases of blood cell contamination
# or simply because they are not of interest.
removing_c = []


In [ ]:
# Checking the required number of samples
if len(adata.obs[sample].cat.categories) < 5:
    print("Unfortunately, it is not possible to perform the analysis using this dataset. For pseudotime analysis, please take a look at the 'general_notebooks' folder.")


### 4.2 Remove specific cell types/cluster 

In [ ]:
for celltype in removing_c:
    adata = adata[adata.obs[clusters] != celltype]

-----------

## 5. Wasserstein distance

To find similarities between cell types and samples, this module uses optimal transport according to [Pyré & Cuturi](https://doi.org/10.48550/arXiv.1803.00567) to calculate the cost matrix and the Wasserstein distance. For this purpose, each sample/patient is modeled as a distribution of cells in clusters (either a specific cell type or condition). Optimal transport is then used to first create the cost matrix to determine the similarities between the clusters and then the transportplan T, assuming mass movement between cluster pairs. Finally, the Wasserstein distance is estimated as the total cost of the optimal transportplan to obtain the similarities between the samples.

For example:
 * short-term events such as cell damage equate to *low costs*
 * while long-term events, e.g., tissue remodeling or scar formation, represent *high costs*

### 5.1 Computing the Wasserstein distance

In [ ]:
pl.tl.wasserstein_distance(adata,
                           emb_matrix=emb_matrix,
                           clusters_col=clusters,
                           sample_col=sample,
                           status=status)

### 5.2 Plotting the Cost matrix and the Wasserstein distance heatmaps

After the computation, we get the heatmaps of the cost matrix and the wasserstein distance. Values may be interpreted as the difference, with smaller values meaning "more similar" and higher values meaning "less similar". The similarities in total are represented with a dendrogram.
    
* **Costmatrix:** Here you can see the similarities between cell types/cell clusters. 
    
* **Wasserstein distance:** This plot shows the similarities between the timepoints/samples. 

In [ ]:
pl.pl.heatmaps(adata, path=settings.figure_dir)

-----------

## 6. Trajectory inference

### 6.1 Diffusion map of Wasserstein Distance

From the Wasserstein distance, we obtain the distance matrix W, which specifies the pairwise distances between all samples and is used to calculate the [diffusion map](https://doi.org/10.1016/j.acha.2006.04.006). In the next step, a path inference algorithm is applied to determine the course of trajectory.

The diffusion map is used for dimensionality reduction or feature extraction by embedding the data into a low-dimensional space using the leading eigenvalues and eigenvectors of a diffusion operator. In this work, two diffusion coordinates are selected to obtain a two-dimensional representation of the data.

The visualization displays the relationship between samples, where similar samples are close and different samples farther apart. It takes the categories from the parameter <code>status</code>. 

In [ ]:
# Set the plot title
plot_title = "zebrafish heart development"

In [ ]:
pl.pl.trajectory(adata,
                 path=settings.figure_dir,
                 plot_title=plot_title)

### 6.2 Estimate the pseudotime

The principal graph is computed on the diffusion map to sort the samples into a timeline(pseudotime) and to estimate the trajectory.

| **Parameter**   | **Description**                                                                                                                                             |
|-----------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **NumNodes**    | The number of nodes within the principal graph. A higher number allows the graph to approximate the time points more closely, which may result in a better estimation. |
| **source_node** | The first timepoint or the patient/sample that determines the start of the trajectory.                                                                      |

<div class="alert alert-block alert-danger">

**For that, it is important to define the start of the trajectory with the help of the parameter <code>source_node</code>.** 

To do this, proceed as follows:
As you can see after starting the plot with the default values below, this function takes the diffusionmap and compute a principal graph with nodes over it (The red points are the points of the diffusionmap). After that, if you have timepoints look at the diffusionmap and find the starting point. Now look at the nodes and find the node near this point and set it as the <code>source_node</code>. In case of patients samples look where you can find the control/healthy samples, take the node at the end of the graph and set this one as the <code>source_node</code>.
    
**Then redo the plot once you changed the <code>source_node</code>.**

</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
NumNodes = 20
source_node = 2

In [ ]:
pl.pl.fit_pricipla_graph(adata,
                         NumNodes=NumNodes,
                         source_node=source_node,
                         path=settings.figure_dir)

-----------

## 7. Cell proportion

Next, we get the cell proportion for each time point using a robust regression model to find groups of cells where the number of cells correlates with the pseudotime progression.

**Heatmap:**
The heatmap shows the cell proportion with the samples on the y-axis sorted by pseudotime (top->bottom = early->late). The x-axis displays the respective groups in the data e.g. cell types.

**Trend plot:**
The trend plot visualizes the cell proportion in total with the regression model. The samples are sorted by pseudotime on the x-axis, and cell coverage is displayed on the y-axis. The line represents the median trend of the sample points.

**Before starting this function, please decide how you want to label the x-axis:**

| **Parameter** | **Description** | **Recommendation** |
|:-----------:|:-------------|:-------------|
|  <code>axis</code>  |  Either <code>categoric</code> or <code>numeric</code>. <br>Edit the line-plot to either show labels (categoric)<br> or numeric values(numeric).  | Use <code>categoric</code> if your samples correspond to timepoints, such as stages of development, <br>and <code>numeric</code> if you have a large number of samples for which it may be unnecessary to display the label.|

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Determine if you have timepoints ("categoric") or samples ("numeric")
axis = "categoric"

In [ ]:
pl.tl.cell_importance(adata,
                      axis=axis,
                      path_plt=settings.figure_dir,
                      path_table=settings.table_dir)

-----------

## 8. Finding gene markers for the clusters

Now that we have obtained an initial overview of cell type progression along the trajectory, we next examine genes whose expression changes continuously and significantly along this trajectory for each cell type.

Three regression models are used for this purpose, with the most significant model being specified for each gene: 

+ linear
+ quadratic
+ quadratic-linear 
 
After running the command, you can find a folder named ‘Markers’ under `../tables/pilot/`. It provides a folder for each cell type. The file `Whole_expressions.csv` contains all statistics associated with genes for that cell type.

<div class="alert alert-block alert-danger">
    
**Note: This is resource-intensive. May take >1hr per cell type/cluster.**
    
</div>

In [ ]:
utils.multiprocessing.mp_adata_first_arg(pl.tl.genes_importance,
                                         adata,
                                         adata.uns['cellnames'],
                                         sample_col=sample,
                                         col_cell=clusters,
                                         plot_genes=False,
                                         path=settings.table_dir)

-----------

## 9. Clustering gene markers by pattern

Since we have found significant markers associated with the trajectory for each cell type, it would be interesting to cluster genes whose patterns are similar to each other. This gives us deeper insight into the gene expression mechanism and allows us to identify which genes may be co-expressed and involved in which signal pathways. 

You can find curves activities' statistical scores that show the FCs of the genes through disease progression in the Markers folder for each cell type.

<div class="alert alert-block alert-danger">
    
**Please clarify below your cluster/cell type by using the parameter <code>cluster</code> and your organism of interest by using the parameter <code>organism</code>. All the following steps will focus on the selected cluster and organism.**
    
</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# The cell type/cluster of interest
cluster = 'Myocardium'

# Set the range of the cluster
scaler_value = 0.4

#### Plot descriptions

1. **Heatmap**: The heatmap shows the expression of genes over time (each row in the heatmap represents a gene), with the samples ordered by pseudotime as the columns. The dendrogram visualizes the relationships between genes based on their expression patterns. A bar displays the range of the clusters with labels on the right of the heatmap.
2. **Expression pattern per cluster**: With the pseudotime on the x-axis and gene expression on the y-axis, the blue line shows  the average expression pattern per cluster overall for the clustered genes, while the blue area indicates the variance. The   larger the area, the more different expression patterns are between the clustered genes.
3. **Top genes per cluster**: The ten most important genes in this cluster are displayed here, with the most important at the bottom and the least important at the top.
4. **Expressionpatterns of the top genes per cluster**: Now this plot takes a further look on the expression pattern of each  top gen. Again on the x-axis we have the pseudotime and on the y-axis the z-scored gene expression. The first dot above the x-axis   are cells with no expression. After that each dot represents a cell that expresses the gene at the timepoint. The orange line displays the expression pattern over time.

In [ ]:
pl.pl.genes_selection_analysis(adata,
                               cluster,
                               scaler_value=scaler_value,
                               path_to_tables=settings.table_dir,
                               path_to_figures=settings.figure_dir)

-----------

## 10 GO enrichment for clustered genes

Here, we utilize the [Enrichr](https://maayanlab.cloud/Enrichr/) tool to identify enriched GO pathways for the clustered genes.

### 10.1 Select library

<div class="alert alert-block alert-danger">

**Since Enrichr is used for this function, the possible organisms are unfortunately limited. The organisms are listed in the table below. Please select the respective organism (<code>organism</code>) and library (<code>dataset</code>) in the input below.**
</div>

**Organism and their ID:** 

| Organism  | Enrichr-ID | Recommended library | More libraries |
|--------|-------------|-------------|-------------|
|*H. sapiens*|"human"| "MSigDB_Hallmark_2020"|https://maayanlab.cloud/Enrichr/#libraries|
|*M. musculus*|"mouse"| "MSigDB_Hallmark_2020"|https://maayanlab.cloud/Enrichr/#libraries|
|*D. rerio*|"fish"| "GO_Biological_Process_2018"|https://maayanlab.cloud/FishEnrichr/#stats|
|*C. elegans*|"worm"| "GO_Biological_Process_2018"|https://maayanlab.cloud/WormEnrichr/#stats|
|*S. cerevisiae*|"yeast"| "GO_Biological_Process_2018"|https://maayanlab.cloud/YeastEnrichr/#stats|
|*D. melanogaster*|"fly"|  "GO_Biological_Process_2018"|https://maayanlab.cloud/FlyEnrichr/#stats|

### Explanation for some Library terms

Below is a brief introduction to other main library categories. Select a library that matches your research question, e.g., *biological process* libraries to investigate regulatory changes. For more information, see [here](https://geneontology.org/docs/ontology-documentation/)

**Hallmark genes** <br>
Hallmarks gene sets summarizes the relevant pathway informations from multiple "founder" sets of the database. As a result of that the gene sets are refined and concise by reducing variation and redundancy. For this "The Molecular Signatures Database" (MSigDB) is used for the organism human and mouse, which provides various gen sets of metabolic diseases and biologic processes.
**Example: MSigDB_Hallmark_2020**

**Molecular Function**  
Molecular-level activities performed by gene products. Molecular function terms describe activities that occur at the molecular level, such as “catalysis” or “transport”. GO molecular function terms represent activities rather than the entities (molecules or complexes) that perform the actions, and do not specify where, when, or in what context the action takes place. Molecular functions generally correspond to activities that can be performed by individual gene products (i.e. a protein or RNA), but some activities are performed by molecular complexes composed of multiple gene products. Examples of broad functional terms are catalytic activity and transporter activity; examples of narrower functional terms are adenylate cyclase activity or Toll-like receptor binding. To avoid confusion between gene product names and their molecular functions, GO molecular functions are often appended with the word “activity” (a protein kinase would have the GO molecular function protein kinase activity). **Examples: GO_Molecular_Function, KEGG**
  
  
**Cellular Component**  
A location, relative to cellular compartments and structures, occupied by a macromolecular machine. There are two ways in which the gene ontology describes locations of gene products: (1) the cellular anatomical entities, in which a gene product carries out a molecular function. Cellular anatomical entities includes cellular structures such as the plasma membrane and the cytoskeleton, as well as membrane-enclosed cellular compartments such as the mitochondrion, and (2) the stable macromolecular complexes of which they are parts, e.g., the clathrin complex. **Examples: GO_Cellular_Component, Anatomy_AutoRIF**

**Biological Process**  
The larger processes, or ‘biological programs’ accomplished by multiple molecular activities. Examples of broad biological process terms are DNA repair or signal transduction. Examples of more specific terms are pyrimidine nucleobase biosynthetic process or glucose transmembrane transport. The 'WikiPathways' database contains detailed descriptions of biological pathways.**Examples: GO_Biological_Process, WikiPathways**
  
**GeneRIF**  
Stands for “Gene Reference Into Functions” and comes from the NCBI database, which provides short, manually curated descriptions of gene functions based on literature references.

**AutoRIF**  
Stands for “Automatically generated GeneRIF.” It is equivalent to GeneRIF, but the descriptions were not compiled manually; instead, they were generated automatically from PubMed abstracts and articles. It is not as accurate as GeneRIF, but it offers more descriptions and is more up to date.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Specific the dataset of interest
dataset = "MSigDB_Hallmark_2020"

# Set the organism of interest
organism = "human"

### 10.2 Identification of pathways

The hallmark-plot shows a heatmap with the clusters on the x-axis and the GO-terms of the selected library on the y-axis.

In [ ]:
dic = {"human": "hsapiens", "fish": "drerio", "mouse": "mmusculus", "worm": "celegans",
       "yeast": "scerevisiae", "fly": "dmelanogaster"}

if organism in dic:
    pl.pl.plot_hallmark_genes_clusters(adata,
                                       cluster, dataset,
                                       path=settings.figure_dir,
                                       organism=organism)
else:
    warn("No analysis possible with your organism of interest")

-----------

## 11. Finding specifc marker

The previous test only finds markers with significant changes over time for a specific cell type. However, it does not consider whether the observation is unique to this cluster and therefore potentially relevant for the cell type differentiation. 

In this section, we aim to identify markers specific to a cell type/cell cluster, that exhibit a different expression pattern compared to other clusters. We use a Wald test that compares the fit of the gene in the cell type/cell cluster with the fit of the gene in other cell types/cell clusters.

You can run this function with all cell types at once, so that the calculation does not have to be performed separately for each cell type. However, the further analysis steps are still based on the cell types previously defined by the parameter <code>cluster</code>.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Number of top genes to consider for each expression pattern
number_genes = 40

# clusters/cell types of interest
cellnames = ['Myocardium', 'Endocardium', 'Epicardium', 'Fibroblasts ']

<div class="alert alert-block alert-danger">

**Note: This is resource-intensive. May take >1hr per cell type/cluster.**

</div>

In [ ]:
pl.tl.gene_cluster_differentiation(adata,
                                   cellnames=cellnames,
                                   number_genes=number_genes,
                                   path=settings.table_dir)

### 11.1 Further filtering based on FC and p-value

In this section, a final selection is made by considering only genes with a FC greater than '0.5' and whose Wald test p-value is less than '0.01'. Below, you can take a closer look at the filtered table. The higher the FC value, the more the expression differs from the other clusters. You can set personalized thresholds using the parameters <code>fc</code> and <code>pvalue</code>.

The filtered table is saved as `gene_clusters_stats_extend_[celltype]_filtered.csv`.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# The number of rows in the table that will be displayed
head_number = 100

# Set the threshold of FC
fc = 0.5

# Set the threshold of p-value
pvalue = 0.01

In [ ]:
df = pl.tl.results_gene_cluster_differentiation(cluster_name=cluster,
                                                path=settings.table_dir,
                                                threshold=fc,
                                                p_value=pvalue)

with pd.option_context("display.max.rows", None, "display.max.columns", None):
    display(df.head(head_number))

-----------

## 12. Expression pattern of specfic genes

### 12.1 Expression pattern over time

Here, we visualize the expression pattern of genes of interest, which may be identified from the table above. The result is a plot similar to the second plot of section 9, where the x-axis displays the pseudotime, and the y-axis the z-scored gene expression . The first dot above the x-axis are cells with no expression. Each point represents a cell that expresses the gene at that timepoint. The orange line shows the expression pattern over time.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

<div class="alert alert-block alert-danger">
    
**Look in the previous table for potential genes of interest.**
    
</div>

In [ ]:
# The gene(s) of interest
gene_list = ['myl7', 'tcap', 'irs1', 'pdk2a', 'bmp4']

In [ ]:
pl.pl.plot_gene_list_pattern(gene_list,
                             cluster,
                             path_to_tables=settings.table_dir,
                             axis=axis)

### 12.2 Comparing expression pattern with other clusters

The plot shows the expression (y-axis) over time (x-axis) of the respective gene within the selected cluster (colored dots and orange line) compared to the average expression of other clusters (grey dots and grey line).

In [ ]:
pl.pl.exploring_specific_genes(adata,
                               cluster_name=cluster,
                               gene_list=gene_list,
                               path_to_tables=settings.table_dir,
                               path_to_figures=settings.figure_dir,
                               axis=axis)

-----------

## 13. Go enrichment for specific markers

Here is the GO enrichment for specific gene markers of the selected cell type. Unlike before [g:Profiler](https://biit.cs.ut.ee/gprofiler/gost) is used instead of Enrichr. It shows which biological functions, signaling pathways, diseases, or regulatory motifs the respective gene set covers. 

The terms shown on the y-axis are ranked according to their significance, with the most significant at the top. The plot is saved in the \"GO\" folder.

<div class="alert alert-block alert-danger">
    
**The parameter <code>organism</code> from section 10 will convert the Enichr-ID to the respective gProfiler_ID if possible. If it isn't listed in the table, you will need to consult the website below to find the ID for your organism of interest.**
    
</div>

**Here are common organism and their gprofil-ID:** 

| Organism  | gprofil-ID |
|--------|-------------|
|human|"hsapiens"|
|mouse|"mmusculus"|
|rat|"rnorvegicus"|
|*C. elegans*|"celegans"|
|zebrafish|"drerio"| 
|*S. cerevisiae*|"scerevisiae"|   
|fruit fly|"dmelanogaster"| 
       
**Organism-ID for gProfiler:**
https://biit.cs.ut.ee/gprofiler/page/organism-list

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Top terms shown for GO enrichment
num_gos = 10
organism_gp = None

In [ ]:
if organism_gp is None and organism not in dic:
    raise ValueError("Invalid organism. Please select an organism from the list above.")

pl.pl.go_enrichment(df,
                    cell_type=cluster,
                    organism=organism_gp if organism_gp else dic[organism],
                    num_gos=num_gos,
                    path_plt=settings.figure_dir,
                    path_table=settings.table_dir)

-----------

## 14. Saving adata

In [ ]:
utils.adata.save_h5ad(adata, "anndata_pilot.h5ad")

In [ ]:
settings.close_logfile()